# 09 gpt-5.1 Reasoning & Verbosity Probe (TEMPORARY)

Quick feasibility notebook. **Delete when done.**

Goal: confirm whether the **`reasoning_effort`** and **`verbosity`** parameters are accepted by the `chat51` deployment, and whether they change the output.

We also check what happens when we pair each of them with an explicit `temperature` value (gpt-5.x reasoning models historically reject custom temperatures).

Uses the official **`openai`** Python SDK pointed at the Azure OpenAI **v1** surface (`/openai/v1/`) with the API key — the documented v1 GA pattern.

## Install the openai SDK

Not in `requirements.txt` for this demo, so install on demand.

In [ ]:
%pip install --quiet "openai>=1.54.0"

## Load environment and create v1 client

Standard `OpenAI` client pointed at `<endpoint>/openai/v1/` with the API key. This is the documented v1 GA pattern — no `api-version` query, no manual headers, no `AzureOpenAI` client.

In [ ]:
import os
from pathlib import Path
from dotenv import load_dotenv
from openai import OpenAI

env_path = Path('../env/.env')
if not env_path.exists():
    raise RuntimeError('env/.env not found. Run 01_deploy_infra.ipynb first.')
load_dotenv(env_path)

AZURE_OPENAI_ENDPOINT = os.getenv('AZURE_OPENAI_ENDPOINT', '').rstrip('/')
AZURE_OPENAI_API_KEY = os.getenv('AZURE_OPENAI_API_KEY', '')
GPT51_DEPLOYMENT = os.getenv('AZURE_OPENAI_SECONDARY_DEPLOYMENT', 'chat51')

if not AZURE_OPENAI_API_KEY:
    raise RuntimeError('AZURE_OPENAI_API_KEY is missing.')

client = OpenAI(
    api_key=AZURE_OPENAI_API_KEY,
    base_url=f'{AZURE_OPENAI_ENDPOINT}/openai/v1/',
)

print(f'Base URL:    {AZURE_OPENAI_ENDPOINT}/openai/v1/')
print(f'Deployment:  {GPT51_DEPLOYMENT}')

## Complex reasoning prompt

A small logic puzzle that benefits from multi-step inference, so reasoning-effort differences become visible.

In [ ]:
SYSTEM = (
    'You are a careful logic solver. Show only the final answer and a short justification (no chain-of-thought dump).'
)

USER = (
    'Five colleagues (Alice, Ben, Cara, Dan, Eve) ranked their preferred onsite day Mon-Fri, each picking a different day.\n'
    'Clues:\n'
    '1. Alice prefers an earlier day than Ben but a later day than Cara.\n'
    '2. Dan does not want Monday or Friday.\n'
    '3. Eve prefers exactly the day immediately after Ben.\n'
    '4. Cara is not on Monday.\n'
    'Question: Which day does each person prefer? Then state how many onsite days fall in the first half of the week (Mon-Wed inclusive).'
)

print(USER)

## Probe runner

Each row is one call. We capture the full underlying error if the parameter combination is rejected, so we can see *why* it failed.

In [ ]:
import json
from openai import BadRequestError, APIStatusError

# Reasoning models count reasoning_tokens against max_completion_tokens.
# Keep this generous so high-effort runs still leave room for a visible answer.
MAX_COMPLETION_TOKENS = 8000

def probe(label: str, **extra):
    """Call gpt-5.1 once with the given extra kwargs and return a summary row."""
    row = {
        'label': label,
        'params': {k: v for k, v in extra.items()},
        'ok': False,
        'error_type': '',
        'error_message': '',
        'finish_reason': '',
        'completion_tokens': None,
        'reasoning_tokens': None,
        'visible_chars': 0,
        'preview': '',
    }
    try:
        resp = client.chat.completions.create(
            model=GPT51_DEPLOYMENT,
            messages=[
                {'role': 'system', 'content': SYSTEM},
                {'role': 'user', 'content': USER},
            ],
            max_completion_tokens=MAX_COMPLETION_TOKENS,
            **extra,
        )
    except BadRequestError as e:
        row['error_type'] = 'BadRequestError'
        row['error_message'] = str(e)
        return row
    except APIStatusError as e:
        row['error_type'] = type(e).__name__
        row['error_message'] = str(e)
        return row

    choice = resp.choices[0]
    text = (choice.message.content or '').strip()
    usage = resp.usage
    details = getattr(usage, 'completion_tokens_details', None)
    row.update({
        'ok': True,
        'finish_reason': choice.finish_reason or '',
        'completion_tokens': getattr(usage, 'completion_tokens', None),
        'reasoning_tokens': getattr(details, 'reasoning_tokens', None) if details else None,
        'visible_chars': len(text),
        'preview': text,
    })
    return row

print('probe() ready')

## Test matrix

Each call tests one parameter combo. We're answering two questions per parameter:
1. Does the deployment **accept** it (no 400)?
2. Does it visibly **change** the output (length, reasoning_tokens, or content)?

Pairings with `temperature` are intentional — gpt-5.x reasoning models typically reject anything other than the default temperature.

In [ ]:
cases = [
    ('baseline (no extras)',                {}),

    ('reasoning_effort=minimal',            {'reasoning_effort': 'minimal'}),
    ('reasoning_effort=low',                {'reasoning_effort': 'low'}),
    ('reasoning_effort=high',               {'reasoning_effort': 'high'}),
    ('reasoning_effort=high + temp=0.7',    {'reasoning_effort': 'high', 'temperature': 0.7}),
    ('reasoning_effort=low  + temp=1.0',    {'reasoning_effort': 'low',  'temperature': 1.0}),

    ('verbosity=low',                       {'verbosity': 'low'}),
    ('verbosity=medium',                    {'verbosity': 'medium'}),
    ('verbosity=high',                      {'verbosity': 'high'}),
    ('verbosity=high + temp=0.7',           {'verbosity': 'high', 'temperature': 0.7}),
    ('verbosity=low  + temp=1.0',           {'verbosity': 'low',  'temperature': 1.0}),

    ('reasoning=high + verbosity=high',     {'reasoning_effort': 'high', 'verbosity': 'high'}),
    ('reasoning=high + verbosity=low',      {'reasoning_effort': 'high', 'verbosity': 'low'}),
]

results = []
for label, extra in cases:
    print(f'--> {label}')
    results.append(probe(label, **extra))

print(f'\nCollected {len(results)} rows.')

## Summary table

Look at:
- `ok` column → was the parameter combo **accepted**?
- `reasoning_tokens` → did `reasoning_effort` actually change internal compute?
- `visible_chars` → did `verbosity` actually change output length?

In [ ]:
from IPython.display import Markdown, display

header = '| label | ok | finish | completion_tok | reasoning_tok | visible_chars | error |'
sep    = '|---|---|---|---|---|---|---|'
lines = [header, sep]
for r in results:
    err = ''
    if not r['ok']:
        # Trim error to keep table readable
        msg = r['error_message'].replace('|', '/').replace('\n', ' ')
        err = f"{r['error_type']}: {msg[:160]}"
    lines.append(
        f"| {r['label']} | {r['ok']} | {r['finish_reason']} | "
        f"{r['completion_tokens']} | {r['reasoning_tokens']} | {r['visible_chars']} | {err} |"
    )
display(Markdown('\n'.join(lines)))

## Inspect the visible answers

Confirm whether different `verbosity` settings actually produce different text (and that `reasoning_effort=high` still answers the puzzle correctly).

In [ ]:
for r in results:
    print('=' * 80)
    print(f"[{r['label']}]  params={r['params']}")
    if not r['ok']:
        print(f"  ERROR ({r['error_type']}): {r['error_message'][:500]}")
        continue
    print(f"  finish={r['finish_reason']}  completion_tok={r['completion_tokens']}  reasoning_tok={r['reasoning_tokens']}  visible_chars={r['visible_chars']}")
    print('  ---')
    print('  ' + r['preview'].replace('\n', '\n  ')[:1500])